# Dataset 4 — Solar Power Generation Data (Kaggle) — Etapa B

Este notebook é a **Etapa B** da análise e é **independente** do notebook da Etapa A: ele não reaproveita nenhuma variável de outra sessão, apenas carrega diretamente o arquivo `amostra_solar.csv`, que foi gerado e exportado pela Etapa A (notebook `Dataset_4_Solar_EtapaA.ipynb`, Seção 3 — Amostragem).

**Pré-requisito:** é necessário que `amostra_solar.csv` já exista nesta mesma pasta (gerado pela Etapa A) antes de executar este notebook.

A partir da amostra, realizamos a renomeação das colunas, a identificação dos períodos de alta geração e a análise de frequência de inversores.

## Carregamento da amostra gerada pela Etapa A

Carregamos diretamente `amostra_solar.csv`, sem depender de nenhuma variável em memória de outro notebook.

In [1]:
import pandas as pd

amostra = pd.read_csv("amostra_solar.csv")

print("Shape da amostra carregada:", amostra.shape)
amostra.head()

Shape da amostra carregada: (13756, 6)


,DATE_TIME,SOURCE_KEY,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD
0,10-06-2020 19:00,uHbuxQJl8lW7ozc,0.000,0.0000,6645.000,7240508.000
1,28-05-2020 00:15,rGa61gmuvPhdLxV,0.000,0.0000,0.000,7206549.000
2,17-05-2020 15:15,sjndEbLyjtCKgGv,7333.875,718.0625,6442.000,7036055.000
3,01-06-2020 04:15,ih0vzX44oOqAx2f,0.000,0.0000,0.000,6308232.000
4,17-05-2020 09:00,z9Y9gH1T5YWrNuG,7665.500,750.4375,961.375,7021701.375


## Seção 4 — Organização (a partir daqui, trabalhamos só com a amostra)

Renomeamos as colunas da amostra para nomes em português, facilitando a leitura das próximas etapas.

In [2]:
amostra_renomeada = amostra.rename(columns={
    "DATE_TIME": "Data_Hora",
    "SOURCE_KEY": "Inversor",
    "DC_POWER": "Potencia_CC",
    "AC_POWER": "Potencia_CA",
    "DAILY_YIELD": "Geracao_Diaria",
    "TOTAL_YIELD": "Geracao_Total",
})

print("Shape da amostra renomeada:", amostra_renomeada.shape)
amostra_renomeada.head()

Shape da amostra renomeada: (13756, 6)


,Data_Hora,Inversor,Potencia_CC,Potencia_CA,Geracao_Diaria,Geracao_Total
0,10-06-2020 19:00,uHbuxQJl8lW7ozc,0.000,0.0000,6645.000,7240508.000
1,28-05-2020 00:15,rGa61gmuvPhdLxV,0.000,0.0000,0.000,7206549.000
2,17-05-2020 15:15,sjndEbLyjtCKgGv,7333.875,718.0625,6442.000,7036055.000
3,01-06-2020 04:15,ih0vzX44oOqAx2f,0.000,0.0000,0.000,6308232.000
4,17-05-2020 09:00,z9Y9gH1T5YWrNuG,7665.500,750.4375,961.375,7021701.375


In [3]:
amostra_renomeada.info()

<class 'pandas.DataFrame'>
RangeIndex: 13756 entries, 0 to 13755
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Data_Hora       13756 non-null  str    
 1   Inversor        13756 non-null  str    
 2   Potencia_CC     13756 non-null  float64
 3   Potencia_CA     13756 non-null  float64
 4   Geracao_Diaria  13756 non-null  float64
 5   Geracao_Total   13756 non-null  float64
dtypes: float64(4), str(2)
memory usage: 644.9 KB


In [4]:
amostra_renomeada.describe()

,Potencia_CC,Potencia_CA,Geracao_Diaria,Geracao_Total
count,13756.000000,13756.000000,13756.000000,1.375600e+04
mean,3135.848747,306.695495,3319.235219,6.978829e+06
std,4023.069798,393.124281,3142.249934,4.162164e+05
min,0.000000,0.000000,0.000000,6.183645e+06
25%,0.000000,0.000000,0.000000,6.509779e+06
50%,370.071429,35.771429,2730.428571,7.146218e+06
75%,6350.558036,622.065625,6293.785714,7.267996e+06
max,14416.142860,1405.585714,9163.000000,7.846821e+06


## Seção 5 — Períodos de alta geração

A partir da amostra renomeada, calculamos a maior `Potencia_CA` registrada, definimos um limiar de 70% desse valor máximo e isolamos os registros de alta geração (acima do limiar).

In [5]:
potencia_ca_max = amostra_renomeada["Potencia_CA"].max()
limiar_70 = potencia_ca_max * 0.70

print(f"Maior Potencia_CA na amostra: {potencia_ca_max}")
print(f"Limiar de 70% do máximo: {limiar_70:.4f}")

Maior Potencia_CA na amostra: 1405.585714
Limiar de 70% do máximo: 983.9100


In [6]:
alta_geracao = amostra_renomeada[amostra_renomeada["Potencia_CA"] > limiar_70].copy()

qtd_alta = len(alta_geracao)
total_amostra = len(amostra_renomeada)
pct_alta = qtd_alta / total_amostra * 100

print(f"Registros de alta geração (Potencia_CA > {limiar_70:.2f}): {qtd_alta}")
print(f"Total de registros na amostra: {total_amostra}")
print(f"Percentual de alta geração: {pct_alta:.2f}%")

alta_geracao.head()

Registros de alta geração (Potencia_CA > 983.91): 1185
Total de registros na amostra: 13756
Percentual de alta geração: 8.61%


,Data_Hora,Inversor,Potencia_CC,Potencia_CA,Geracao_Diaria,Geracao_Total
11,26-05-2020 12:15,bvBOhCH3iADSZry,11830.14286,1153.828571,4028.142857,6396364.143
46,15-06-2020 14:30,wCURE6d3bPkepu2,10571.28571,1032.071429,5366.571429,7015064.571
48,25-05-2020 10:15,z9Y9gH1T5YWrNuG,11050.14286,1079.857143,2174.285714,7082812.286
72,13-06-2020 11:30,7JYdWkrLSPkdwr4,11094.50000,1082.437500,3570.250000,7815987.250
76,07-06-2020 13:00,VHMLBKoKgIrUVDU,12663.42857,1234.685714,4976.714286,7382965.714


## Seção 6 — Frequência de inversores

Analisamos, dentro do subconjunto de alta geração, quais inversores (`Inversor`, correspondente ao `SOURCE_KEY` original) aparecem com maior frequência.

In [7]:
contagem_inversores = alta_geracao["Inversor"].value_counts()
print("Frequência dos inversores nos registros de alta geração:")
contagem_inversores

Frequência dos inversores nos registros de alta geração:


Inversor
adLQvlD726eNBSB    68
McdE0feGgRqW7Ca    65
ZnxXDlPa8U1GXgE    63
pkci93gMrogZuBj    62
ZoEaEvLYb1n2sOq    61
1IF53ai7Xc0U56Y    59
YxYtjZvoooNbGkE    58
sjndEbLyjtCKgGv    57
z9Y9gH1T5YWrNuG    56
uHbuxQJl8lW7ozc    55
zVJPv84UY57bAof    54
ih0vzX44oOqAx2f    53
zBIq5rxdHJRwDNY    53
WRmjgnKYAwPKWDb    52
VHMLBKoKgIrUVDU    51
3PZuoBAID5Wc2HD    51
iCRJl6heRkivqQ3    51
wCURE6d3bPkepu2    49
7JYdWkrLSPkdwr4    43
rGa61gmuvPhdLxV    43
1BY6WEcLGh8j5v7    42
bvBOhCH3iADSZry    39
Name: count, dtype: int64

In [8]:
inversor_mais_frequente = contagem_inversores.index[0]
qtd_mais_frequente = contagem_inversores.iloc[0]

print(f"Inversor mais frequente nos períodos de alta geração: {inversor_mais_frequente}")
print(f"Número de aparições: {qtd_mais_frequente}")

Inversor mais frequente nos períodos de alta geração: adLQvlD726eNBSB
Número de aparições: 68


### Interpretação cautelosa do resultado

O inversor identificado como mais frequente aparece com mais registros dentro do recorte de alta geração (`Potencia_CA` acima de 70% do máximo observado na amostra) **em relação aos demais inversores da mesma amostra**. Isso indica apenas que, nas linhas amostradas, esse inversor esteve associado a momentos de potência elevada com maior frequência relativa — **não** que ele seja necessariamente o inversor "melhor" ou de maior desempenho da planta.

Existem diversos fatores que podem explicar essa frequência maior sem qualquer relação com a qualidade do equipamento: por exemplo, o inversor pode estar fisicamente ligado a um conjunto de painéis com melhor orientação solar, menor sombreamento, temperatura ambiente mais favorável no momento das leituras, ou simplesmente ter mais registros amostrados por acaso (já que a amostragem é aleatória e não estratificada por inversor). Também não é possível descartar variações de irradiância, nebulosidade ou temperatura ao longo do dia que afetam todos os inversores da planta simultaneamente.

Para investigar a causa real por trás dessa frequência — e assim afirmar algo sobre desempenho comparativo entre inversores — seria necessário cruzar esses dados com o arquivo de sensores climáticos (irradiância, temperatura do módulo, temperatura ambiente), o que está **fora do escopo** desta atividade, que trabalha exclusivamente com o arquivo de geração da planta.

## Resumo dos principais números

Célula abaixo consolida os principais resultados numéricos obtidos nesta análise.

In [9]:
print("="*60)
print("RESUMO DOS PRINCIPAIS NÚMEROS")
print("="*60)
print(f"Potência CA máxima (amostra): {potencia_ca_max}")
print(f"Limiar de 70% do máximo: {limiar_70:.4f}")
print(f"Registros de alta geração: {qtd_alta} ({pct_alta:.2f}% da amostra)")
print(f"Inversor mais frequente em alta geração: {inversor_mais_frequente} ({qtd_mais_frequente} ocorrências)")
print("="*60)

RESUMO DOS PRINCIPAIS NÚMEROS
Potência CA máxima (amostra): 1405.585714
Limiar de 70% do máximo: 983.9100
Registros de alta geração: 1185 (8.61% da amostra)
Inversor mais frequente em alta geração: adLQvlD726eNBSB (68 ocorrências)
